Library

In [33]:
import os
import pandas as pd
import pypdf
from pypdf import PdfReader
import pdfplumber
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


Load Text Data

In [75]:
from pathlib import Path

try:
    from pypdf import PdfReader
except Exception:
    PdfReader = None

personal_path = r"C:\Users\nhipk\OneDrive\0. Uni2024\Sem4\ML for Economists\FinalProject" #Edit your personal path here
folder_path = os.path.join(personal_path, "TermpaperFilesILIAS", "C_BundesbankMonthlyReports")
pdf_files = list(sorted(Path(folder_path).glob("*.pdf")))


In [79]:
def load_pdfs_from_folder(folder_path):
    """Read all PDF files from a specified folder, skip the first 4 and last 4 pages, and extract text."""
    data = []

    # Check if the target folder exists
    if not os.path.exists(folder_path):
        print(f"Error: The directory '{folder_path}' does not exist.")
        return None

    # Iterate through all items within the folder
    for filename in os.listdir(folder_path):
        # Process only files with a .pdf extension (case-insensitive)
        if filename.lower().endswith(".pdf"):
            file_path = os.path.join(folder_path, filename)
            print(f"Processing: {filename}...")

            try:
                # Initialize the PDF reader
                reader = PdfReader(file_path)
                total_pages = len(reader.pages)
                
                # Safe check: Only process if the file has more than 8 pages
                if total_pages > 8:
                    full_text = ""
                    
                    # Slice the pages to skip the first 4 and last 4 pages
                    target_pages = reader.pages[4:-4]
                    
                    # Loop through the sliced pages to extract text content
                    for page in target_pages:
                        text = page.extract_text()
                        if text:
                            full_text += text + "\n"

                    # Append the filename and extracted text metadata to the list
                    data.append({"file_name": filename, "text": full_text.strip()})
                else:
                    print(f"⚠️ Skipped '{filename}': File only has {total_pages} pages (too short to drop 4 start/end pages).")

            except Exception as e:
                print(f"Error reading file {filename}: {e}")

    # Convert the extracted data into a pandas DataFrame for seamless NLP/Sentiment Analysis
    df = pd.DataFrame(data)
    return df

# Execute the extraction process
df_sentiment = load_pdfs_from_folder(folder_path)

# Display a preview of the resulting DataFrame
if df_sentiment is not None and not df_sentiment.empty:
    print(f"\nSuccessfully loaded {len(df_sentiment)} files!")
    display(df_sentiment)
else:
    print("\nNo PDF files found or the folder is empty.")

Processing: 2009-12-monatsbericht-data.pdf...
Processing: 2010-12-monatsbericht-data.pdf...
Processing: 2011-12-monatsbericht-data.pdf...
Processing: 2012-12-monatsbericht-data.pdf...
Processing: 2013-12-monatsbericht-data.pdf...
Processing: 2014-12-monatsbericht-data.pdf...
Processing: 2015-12-monatsbericht-data.pdf...
Processing: 2016-12-monatsbericht-data.pdf...
Processing: 2017-12-monatsbericht-data.pdf...
Processing: 2018-12-monatsbericht-data.pdf...
Processing: 2019-12-monatsbericht-data.pdf...
Processing: 2020-12-monatsbericht-data.pdf...
Processing: 2021-12-monatsbericht-data.pdf...
Processing: 2022-12-monatsbericht-data.pdf...
Processing: 2023-12-monatsbericht-data.pdf...

Successfully loaded 15 files!


,file_name,text
0,2009-12-monatsbericht-data.pdf,DEUTSCHE\nBUNDESBANK\nMonthly Report\nDecember...
1,2010-12-monatsbericht-data.pdf,DEUTSCHE\nBUNDESBANK\nMonthly Report\nDecember...
2,2011-12-monatsbericht-data.pdf,Commentaries Economic conditions\nUnderlying t...
3,2012-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
4,2013-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
5,2014-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
6,2015-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
7,2016-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
8,2017-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
9,2018-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...


Process Data

In [80]:
# Add date column as key
df_sentiment["report_date"] = (
    df_sentiment["file_name"].str[:7]
    )
df_sentiment["word_count"] = df_sentiment["text"].str.split().str.len()

display(df_sentiment)

,file_name,text,report_date,word_count
0,2009-12-monatsbericht-data.pdf,DEUTSCHE\nBUNDESBANK\nMonthly Report\nDecember...,2009-12,100371
1,2010-12-monatsbericht-data.pdf,DEUTSCHE\nBUNDESBANK\nMonthly Report\nDecember...,2010-12,102536
2,2011-12-monatsbericht-data.pdf,Commentaries Economic conditions\nUnderlying t...,2011-12,113864
3,2012-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2012-12,119657
4,2013-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2013-12,113182
5,2014-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2014-12,105413
6,2015-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2015-12,112959
7,2016-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2016-12,120586
8,2017-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2017-12,127829
9,2018-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2018-12,116731


In [81]:
# Stop words and Custom Words
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

# Load standard English stop words
stop_words = set(stopwords.words("english"))
print(stop_words)

# Create a custom dictionary of words
custom_stop_words = {
   "report"
   , "monthly report"
   , "bundesbank"
   , "deutsche"
   , "german"
   , "central"
   , "bank"
   , "european"
   , "euro"
   , "zone"
   , "europe"
   , "european central bank"
   , "ecb"
   , "monetary policy"
   , "monetary policy report"
}

# Add the default stop words as well
custom_stop_words = custom_stop_words.union(stop_words)

{'what', 'shan', 'on', "weren't", 'don', 'while', 'should', 'to', "i'd", 'ain', "they'll", 'your', "he'd", 'did', 'myself', 'was', 'having', "wasn't", "you'll", 'over', 'does', 'against', 'o', 'll', "wouldn't", 'some', 'had', 'she', "i'm", 'up', 'where', "they'd", "we're", 'the', 'few', 'be', 'further', "mightn't", "she's", 've', 'once', 'how', 'it', 'an', 'didn', 'is', 'i', 'other', 'as', 'wasn', 'mightn', 's', "it'd", 'itself', "didn't", 'ours', 'yourselves', 'down', "she'll", 'after', 'each', 'nor', "doesn't", "aren't", 'there', "hasn't", 'were', "needn't", "we've", "he's", 'who', 'here', 'd', "mustn't", 'that', 'will', 'and', 'during', 'most', "you'd", 'being', 'yours', 'not', 'whom', 'isn', 'our', 'just', 'do', 'at', 'which', 'won', 'if', 'hasn', "won't", 'any', "you've", "haven't", 'doesn', 'him', "don't", 'out', "it'll", 'too', "shouldn't", 'he', 'am', "they've", "she'd", "shan't", 'my', 'its', "we'd", 'in', "i've", 'so', 'wouldn', 'because', 'has', 'only', 'between', 'haven', '

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nhipk\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\nhipk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nhipk\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nhipk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [83]:
# Remove Stop Words from the text column

# Tokenize text (break it down into individual words)
df_sentiment['cleaned_text'] = df_sentiment['text'].str.lower()
df_sentiment['tokens'] = df_sentiment['cleaned_text'].apply(word_tokenize)


# Keep only words that are not stop words and custom_words

df_sentiment['cleaned_text'] = df_sentiment['cleaned_text'].apply(
    lambda x: ' '.join(
        word for word in word_tokenize(x) if word not in custom_stop_words
    )
)
df_sentiment["word_count"] = df_sentiment["cleaned_text"].str.split().str.len()
display(df_sentiment)

,file_name,text,report_date,word_count,cleaned_text,tokens
0,2009-12-monatsbericht-data.pdf,DEUTSCHE\nBUNDESBANK\nMonthly Report\nDecember...,2009-12,89886,monthly december 2009 5 commentaries economic ...,"[deutsche, bundesbank, monthly, report, decemb..."
1,2010-12-monatsbericht-data.pdf,DEUTSCHE\nBUNDESBANK\nMonthly Report\nDecember...,2010-12,92174,monthly december 2010 5 commentaries economic ...,"[deutsche, bundesbank, monthly, report, decemb..."
2,2011-12-monatsbericht-data.pdf,Commentaries Economic conditions\nUnderlying t...,2011-12,102811,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin..."
3,2012-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2012-12,108630,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin..."
4,2013-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2013-12,100063,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin..."
5,2014-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2014-12,95571,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin..."
6,2015-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2015-12,101370,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin..."
7,2016-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2016-12,108127,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin..."
8,2017-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2017-12,113288,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin..."
9,2018-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2018-12,104866,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin..."
